# Gemma 3 12B — Merge LoRA + Export for TRT-LLM INT4

**Purpose**: Merge your trained LoRA adapter with the base model, then export for TensorRT-LLM INT4 engine build.

**What you need**: Upload `gemma3-ckpt3750vlmtrained/` (your LoRA adapter from training)

**What you get**: Merged 16-bit model (~24GB) ready for INT4 TRT engine conversion

**Hardware**: A100 (40GB) recommended. T4 (15GB) works but slower.

**Time**: ~30 minutes total

---

## Your LoRA Adapter Details
- **Base model**: `unsloth/gemma-3-12b-it-unsloth-bnb-4bit`
- **Architecture**: `Gemma3ForConditionalGeneration` (text + vision)
- **LoRA**: r=16, alpha=32, rslora=True, 7 target modules
- **Training**: 3750 steps, 2 epochs, checkpoint from March 8 2026
- **Adapter size**: ~262MB (adapter_model.safetensors)

## 1. Install Dependencies

In [ ]:
!pip uninstall unsloth -y
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install bitsandbytes accelerate peft transformers huggingface_hub

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram:.1f} GB")

## 2. Upload LoRA Adapter

**Option A**: Upload from local machine (upload the folder contents)

**Option B**: Upload from Google Drive (if you saved it there)

**Option C**: Upload the zip file and extract

In [ ]:
import os
from pathlib import Path

# === CHOOSE YOUR UPLOAD METHOD ===

# Option A: Upload adapter files directly
# Uncomment and run:
# from google.colab import files
# os.makedirs('gemma3-ckpt3750vlmtrained', exist_ok=True)
# uploaded = files.upload()  # Select: adapter_config.json, adapter_model.safetensors, tokenizer.json, tokenizer_config.json
# for name, data in uploaded.items():
#     with open(f'gemma3-ckpt3750vlmtrained/{name}', 'wb') as f:
#         f.write(data)

# Option B: From Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/gemma3-ckpt3750vlmtrained ./

# Option C: Upload zip and extract
from google.colab import files
print("Upload gemma3-ckpt3750vlmtrained.zip (or gemma3-legal-lora-backup.zip)")
uploaded = files.upload()
for name in uploaded:
    if name.endswith('.zip'):
        !unzip -o {name}
        print(f"Extracted: {name}")

# Verify adapter exists
ADAPTER_DIR = None
for candidate in ['gemma3-ckpt3750vlmtrained', 'gemma3-legal-lora', 'gemma3-12b-legal-multimodal-lora']:
    if Path(candidate).exists() and (Path(candidate) / 'adapter_config.json').exists():
        ADAPTER_DIR = candidate
        break

if ADAPTER_DIR:
    print(f"\nAdapter found: {ADAPTER_DIR}/")
    for f in sorted(Path(ADAPTER_DIR).iterdir()):
        size = f.stat().st_size / (1024**2)
        print(f"  {f.name:40s} ({size:>6.1f} MB)")
else:
    print("ERROR: No adapter directory found!")
    print("Expected: adapter_config.json + adapter_model.safetensors")

## 3. Load Base Model + LoRA Adapter

In [ ]:
from unsloth import FastVisionModel
import json

# Read adapter config to get base model name
with open(f'{ADAPTER_DIR}/adapter_config.json') as f:
    adapter_config = json.load(f)

BASE_MODEL = adapter_config['base_model_name_or_path']
print(f"Base model: {BASE_MODEL}")
print(f"LoRA rank: {adapter_config['r']}")
print(f"LoRA alpha: {adapter_config['lora_alpha']}")
print(f"Architecture: {adapter_config['auto_mapping']['base_model_class']}")
print(f"\nLoading base model...")

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

print(f"\nLoading LoRA adapter from {ADAPTER_DIR}...")
from peft import PeftModel
model = PeftModel.from_pretrained(model, ADAPTER_DIR)

print(f"\nBase model + LoRA adapter loaded successfully")

## 4. Quick Inference Test (Optional)

Verify the adapter works before merging.

In [ ]:
FastVisionModel.for_inference(model)

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

test_prompts = [
    "What is the legal standard for admissibility of evidence under the Federal Rules of Evidence?",
    "Explain how Svelte 5 runes ($state, $derived, $effect) replace Svelte 4 stores.",
]

for prompt in test_prompts:
    print(f"\n{'='*70}")
    print(f"Q: {prompt}")
    print(f"{'='*70}")
    
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    model.generate(
        input_ids=inputs, streamer=streamer,
        max_new_tokens=200, temperature=0.7, top_p=0.9, use_cache=True
    )
    print()

## 5. Merge LoRA into Base Model (16-bit)

This creates a standalone model with LoRA weights baked in.
Output: ~24GB for Gemma 3 12B in 16-bit.

In [ ]:
MERGED_DIR = "gemma3-12b-legal-merged-16bit"

print(f"Merging LoRA weights into base model...")
print(f"Output: {MERGED_DIR}/ (~24 GB)")
print(f"This takes 10-15 minutes on A100.\n")

model.save_pretrained_merged(
    MERGED_DIR,
    tokenizer,
    save_method="merged_16bit"
)

# Verify output
merged_path = Path(MERGED_DIR)
total_gb = sum(f.stat().st_size for f in merged_path.rglob('*')) / (1024**3)
safetensor_files = list(merged_path.glob('*.safetensors'))

print(f"\nMerged model saved: {MERGED_DIR}/")
print(f"Total size: {total_gb:.1f} GB")
print(f"Safetensor shards: {len(safetensor_files)}")
for f in sorted(safetensor_files):
    print(f"  {f.name}: {f.stat().st_size / (1024**3):.1f} GB")

## 6. Verify Merged Model Config

Check that the merged model has the right architecture for TRT-LLM conversion.

In [ ]:
import json

config_path = Path(MERGED_DIR) / 'config.json'
if config_path.exists():
    with open(config_path) as f:
        config = json.load(f)
    
    print("Merged Model Config:")
    print(f"  Architecture: {config.get('architectures', ['?'])[0]}")
    print(f"  Hidden size: {config.get('hidden_size', '?')}")
    print(f"  Num layers: {config.get('num_hidden_layers', '?')}")
    print(f"  Num heads: {config.get('num_attention_heads', '?')}")
    print(f"  Vocab size: {config.get('vocab_size', '?')}")
    print(f"  Max position: {config.get('max_position_embeddings', '?')}")
    
    # Check for VLM components
    arch = config.get('architectures', [''])[0]
    if 'Conditional' in arch:
        print(f"\n  VLM: YES (Gemma3ForConditionalGeneration)")
        print(f"  Text decoder: hidden_size={config.get('text_config', {}).get('hidden_size', config.get('hidden_size', '?'))}")
        if 'vision_config' in config:
            vc = config['vision_config']
            print(f"  Vision encoder: {vc.get('model_type', '?')} ({vc.get('hidden_size', '?')}-dim)")
    else:
        print(f"\n  VLM: NO (text-only: {arch})")
    
    print(f"\n  TRT-LLM conversion: python convert_checkpoint.py")
    print(f"    --model_dir {MERGED_DIR}")
    print(f"    --output_dir trt_checkpoint")
    print(f"    --dtype float16")
else:
    print("WARNING: config.json not found in merged model!")

## 7. Save to Google Drive + Download

The merged model is ~24GB. Options:
- **Google Drive**: Copy there, then download from drive.google.com
- **Direct download**: Works but may timeout for large files
- **HuggingFace Hub**: Push to private repo for easy pulling later

In [ ]:
# === OPTION A: Save to Google Drive (RECOMMENDED for 24GB) ===
from google.colab import drive
drive.mount('/content/drive')

import shutil
drive_dest = '/content/drive/MyDrive/gemma3-12b-legal-merged-16bit'

print(f"Copying to Google Drive: {drive_dest}")
print("This takes 10-20 minutes for ~24GB...")

if os.path.exists(drive_dest):
    shutil.rmtree(drive_dest)
shutil.copytree(MERGED_DIR, drive_dest)

print(f"\nSaved to Google Drive!")
print(f"Download from: https://drive.google.com/")
print(f"Path: My Drive / gemma3-12b-legal-merged-16bit/")

In [ ]:
# === OPTION B: Push to HuggingFace Hub (private repo) ===
# Uncomment to use:

# from huggingface_hub import notebook_login
# notebook_login()
#
# HF_REPO = "YOUR_USERNAME/gemma3-12b-legal-merged"  # Change this!
#
# model.push_to_hub_merged(
#     HF_REPO,
#     tokenizer,
#     save_method="merged_16bit",
#     private=True,
# )
#
# print(f"Pushed to: https://huggingface.co/{HF_REPO}")
# print(f"Pull locally: git clone https://huggingface.co/{HF_REPO}")

## 8. Next Steps (On Your Local Machine)

After downloading the merged model (~24GB):

### Text Inference: TRT-LLM INT4 Engine
```bash
# 1. Place merged model
mkdir -p ~/gemma3-12b-legal-merged-16bit
# Copy downloaded files here

# 2. Build INT4 engine inside Docker container
cd ~/Videos/deeds-web-app
bash scripts/build-trt-engine-in-container.sh

# 3. Start Triton
docker compose -f docker-compose.triton.yml up -d
```

### Vision Inference: PyTorch in TRT-LLM Container
The TRT-LLM v0.21.0 container has PyTorch + transformers built-in.
For image queries, load the merged model via PyTorch (on-demand),
process with SigLIP vision encoder, then unload.

```python
# Inside TRT-LLM container:
from transformers import AutoModelForCausalLM, AutoProcessor
model = AutoModelForCausalLM.from_pretrained(
    '/models/gemma3-12b-legal-merged-16bit',
    torch_dtype=torch.float16,
    device_map='auto'
)
# Process image + text prompt via VLM
# Then unload to free VRAM for TRT engine
del model; torch.cuda.empty_cache()
```

### Architecture: Text/Vision Switcher
```
Request → /api/ai/tensorrt/
  ├── text-only → TRT INT4 Engine (fast, persistent, ~6.5GB)
  └── image+text → PyTorch VLM (on-demand, load/unload, ~24GB→offload)
```

Your SvelteKit API route at `/api/ai/tensorrt/vlm/+server.ts` already
implements this pattern with GPU lease acquisition and model load/unload.